#SE NAO TIVER ENV RODA ESSE

In [142]:

# %conda create --name IBD --file environment.yml

# Importando as bibliotecas e também lendo o arquivo excel:

In [143]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import sqlite3
import re
import os
import datetime
from verify_form import *
DB_FILE = 'gas_data.db' # Para ler o banco de dados SQLite
XLSX_FILE = 'gn_marco_2025.xlsx' # Para ler o arquivo XLSX


print(f"Iniciando a extração do arquivo '{XLSX_FILE}'...")
try:
    df_raw = pd.read_excel(XLSX_FILE)
except FileNotFoundError:
    print(f"ERRO: O arquivo '{XLSX_FILE}' não foi encontrado.")
    print("Por favor, certifique-se de que o arquivo está no diretório correto.")
    raise

print("Extração concluída.")

Iniciando a extração do arquivo 'gn_marco_2025.xlsx'...
Extração concluída.


# Normalização:

In [144]:
print("Iniciando a transformação dos dados...")

# Mapeamento de colunas para nomes mais curtos e compatíveis com SQL
column_mapping = {
    'Código da Instalação de Transporte': 'codigo_instalacao_transporte',
    'Nome da Instalação de Transporte': 'nome_instalacao_transporte',
    'Nome da Instalação de Gasoduto': 'nome_instalacao_gasoduto',
    'Código da Instalação de Gasoduto': 'codigo_instalacao_gasoduto',
    'Tipo da instalação de Gasoduto': 'tipo_instalacao',
    'Nome do Município da Instalação de Gasoduto': 'municipio',
    'Nome da UF da Instalação de Gasoduto': 'uf',
    'Nome do Operador da instalação de Gasoduto': 'nome_operador',
    'Código do Operador da Instalação de Gasoduto': 'codigo_operador',
    'Nome do Carregador que usa a Instalação de Gasoduto': 'nome_carregador',
    'Código do Carregador que usa a Instalação de Gasoduto': 'codigo_carregador',
    'Nome do Contrato da Instalação de Gasoduto': 'nome_contrato',
    'Nome da Variável': 'variavel_completa'
}
df = df_raw.rename(columns=column_mapping)

Iniciando a transformação dos dados...


# UNPIVOT

In [145]:
# Identificar colunas de data para a operação de "unpivot"
date_columns = [col for col in df.columns if isinstance(col, datetime.datetime)]
id_vars = list(column_mapping.values())

#UNPIVOT (transformar de formato largo para longo)
df_long = pd.melt(df, id_vars=id_vars, value_vars=date_columns,
                  var_name='data_medicao', value_name='valor')

#convert data
df_long.replace('#N/D', np.nan, inplace=True)
df_long['valor'] = pd.to_numeric(df_long['valor'], errors='coerce')
df_long.dropna(subset=['valor'], inplace=True)
df_long['data_medicao'] = pd.to_datetime(df_long['data_medicao'])

#Extract variable name and unit
var_regex = re.compile(r'^(.*?)\s*\((.*)\)$')
extracted_vars = df_long['variavel_completa'].str.extract(var_regex)

df_long['nome_variavel'] = extracted_vars[0].str.strip()
df_long['unidade_medida'] = extracted_vars[1].str.strip()
df_long['nome_variavel'].fillna(df_long['variavel_completa'], inplace=True)
df_long['unidade_medida'].fillna('N/A', inplace=True)


/tmp/ipykernel_56817/2559230385.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_long.replace('#N/D', np.nan, inplace=True)
/tmp/ipykernel_56817/2559230385.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_long['nome_variavel'].fillna(df_long['variavel_completa'], inplace=True)
/tmp/ipy

## Cria DB

In [146]:
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = 0;") # Habilitar restrições de chave estrangeira

# DDL - Data Definition Language (Comandos para criar as tabelas)
ddl_scripts = """
CREATE TABLE Operador (
    codigo_operador INTEGER PRIMARY KEY,
    nome_operador TEXT NOT NULL UNIQUE
);

CREATE TABLE Carregador (
    codigo_carregador INTEGER PRIMARY KEY,
    nome_carregador TEXT NOT NULL UNIQUE
);

CREATE TABLE Variavel (
    id_variavel INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_variavel TEXT NOT NULL,
    unidade_medida TEXT NOT NULL,
    CONSTRAINT uq_variavel_nome_unidade UNIQUE (nome_variavel, unidade_medida)
);

CREATE TABLE Instalacao_Transporte (
    codigo_instalacao_transporte INTEGER PRIMARY KEY,
    nome_instalacao_transporte TEXT NOT NULL UNIQUE
);

CREATE TABLE Instalacao_Gasoduto (
    codigo_instalacao_gasoduto INTEGER PRIMARY KEY,
    nome_instalacao_gasoduto TEXT NOT NULL,
    tipo_instalacao TEXT NOT NULL,
    municipio TEXT NOT NULL,
    uf TEXT NOT NULL,
    codigo_instalacao_transporte INTEGER NOT NULL,
    FOREIGN KEY (codigo_instalacao_transporte) REFERENCES Instalacao_Transporte(codigo_instalacao_transporte)
);


CREATE TABLE Contrato (
    codigo_instalacao_transporte INTEGER NOT NULL,
    codigo_carregador INTEGER NOT NULL,
    nome_contrato TEXT,
    FOREIGN KEY (codigo_instalacao_transporte) REFERENCES Instalacao_Transporte(codigo_instalacao_transporte),
    FOREIGN KEY (codigo_carregador) REFERENCES Carregador(codigo_carregador),
    PRIMARY KEY (codigo_instalacao_transporte, codigo_carregador,nome_contrato)
);

CREATE TABLE Medicao (
    id_medicao INTEGER PRIMARY KEY AUTOINCREMENT,
    data_medicao DATE NOT NULL,
    valor REAL NOT NULL,
    id_variavel INTEGER NOT NULL,
    codigo_carregador INTEGER NOT NULL,
    codigo_instalacao_transporte INTEGER NOT NULL,
    codigo_instalacao_gasoduto INTEGER NULL,
    FOREIGN KEY (id_variavel) REFERENCES Variavel(id_variavel),
    FOREIGN KEY (codigo_carregador) REFERENCES Carregador(codigo_carregador),
    FOREIGN KEY (codigo_instalacao_transporte) REFERENCES Instalacao_Transporte(codigo_instalacao_transporte),
    FOREIGN KEY (codigo_instalacao_gasoduto) REFERENCES Instalacao_Gasoduto(codigo_instalacao_gasoduto),
    CONSTRAINT uq_medicao_unica UNIQUE (data_medicao, id_variavel, codigo_carregador, codigo_instalacao_transporte, codigo_instalacao_gasoduto)
);
"""

cursor.executescript(ddl_scripts)

## Popula operadores:

In [147]:
# Operador
df_operador = df_long[['codigo_operador', 'nome_operador']].drop_duplicates().dropna(subset=['codigo_operador'])
df_operador.to_sql('Operador', conn, if_exists='append', index=False)
df_operador

,codigo_operador,nome_operador
0,1717813,Gasocidente do Mato Grosso Ltda - GOM
34,3004992714,Nova Transportadora do Sudeste S.A. - NTS
7338,3006248349,Transportadora Associada de Gás S.A. - TAG
10148,5,Transportadora Brasileira Gasoduto Bolívia-Bra...
11105,3003146349,Transportadora Sulbrasileira de Gás S.A. - TSB


In [148]:
verify_database_normalization(DB_FILE,'Operador')

The table 'Operador' is in 1NF.
The table 'Operador' is in 2NF.
The table 'Operador' is in 3NF.


{'Operador': {'1NF': True, '2NF': True, '3NF': True}}

## Criando o dataset de carregadores:

In [149]:
df_carregador = df_long[['codigo_carregador', 'nome_carregador']] \
    .dropna(subset=['codigo_carregador', 'nome_carregador']) \
    .drop_duplicates(subset=['codigo_carregador'])

df_carregador.to_sql('Carregador', conn, if_exists='append', index=False)
df_carregador.head()

,codigo_carregador,nome_carregador
0,1.645009e+06,AMBAR
17,6.023921e+06,MTGAS
34,3.300017e+07,Petróleo Brasileiro S.A. - PETROBRAS
1120,2.033043e+09,Companhia Siderúrgica Nacional
2148,2.232025e+06,BRAVA


In [150]:
verify_database_normalization(DB_FILE,'Carregador')

The table 'Carregador' is in 1NF.
The table 'Carregador' is in 2NF.
The table 'Carregador' is in 3NF.


{'Carregador': {'1NF': True, '2NF': True, '3NF': True}}

## Criando o dataset de variaveis:

In [151]:
df_variavel = df_long[['nome_variavel', 'unidade_medida']] \
    .dropna(subset=['nome_variavel', 'unidade_medida']) \
    .drop_duplicates()

df_variavel.to_sql('Variavel', conn, if_exists='append', index=False)
df_variavel.head()

,nome_variavel,unidade_medida
0,Gás de Uso no Sistema,mil m³
1,Gás não contado,mil m³
2,Perdas Operacionais,mil m³
3,Perdas Extraordinárias,mil m³
4,Desequilíbrio Diário,mil m³


In [152]:
verify_database_normalization(DB_FILE,'Variavel')

The table 'Variavel' is in 1NF.
The table 'Variavel' is in 2NF.
The table 'Variavel' is in 3NF.


{'Variavel': {'1NF': True, '2NF': True, '3NF': True}}

## Criando o dataset da instalação de transporte:

In [153]:
df_instalacao_transporte = df_long[['codigo_instalacao_transporte', 'nome_instalacao_transporte']] \
    .dropna(subset=['codigo_instalacao_transporte', 'nome_instalacao_transporte']) \
    .drop_duplicates(subset=['nome_instalacao_transporte'])

# Normalize case and strip spaces
df_instalacao_transporte['nome_instalacao_transporte'] = df_instalacao_transporte['nome_instalacao_transporte'].str.strip().str.lower()

df_instalacao_transporte.to_sql('Instalacao_Transporte', conn, if_exists='append', index=False)
df_instalacao_transporte.head()

,codigo_instalacao_transporte,nome_instalacao_transporte
0,700521,bolívia - mato grosso lateral cuiabá
34,700543,gasduc iii
86,700525,paulínia-jacutinga
103,700540,gastau
125,514180,anel de gás


In [154]:
verify_database_normalization(DB_FILE,'Instalacao_Transporte')

The table 'Instalacao_Transporte' is in 1NF.
The table 'Instalacao_Transporte' is in 2NF.
The table 'Instalacao_Transporte' is in 3NF.


{'Instalacao_Transporte': {'1NF': True, '2NF': True, '3NF': True}}

## Criando o dataset da instalação de gasoduto

In [155]:
df[['codigo_instalacao_gasoduto', 'codigo_operador']].drop_duplicates(subset=['codigo_instalacao_gasoduto', 'codigo_operador'])

,codigo_instalacao_gasoduto,codigo_operador
0,NaN,1717813
7,111567.0,1717813
12,111565.0,1717813
29,111566.0,1717813
34,NaN,3004992714
...,...,...
11105,NaN,3003146349
11112,212585.0,3003146349
11117,212586.0,3003146349
11129,212587.0,3003146349


In [156]:
df_instalacao_gasoduto = df_long[['codigo_instalacao_gasoduto', 'nome_instalacao_gasoduto', 'tipo_instalacao', 'municipio', 'uf', 'codigo_instalacao_transporte']] \
    .dropna(subset=['codigo_instalacao_gasoduto']) \
    .drop_duplicates()
# Filtra para inserir apenas os gasodutos que ainda não existem

df_instalacao_gasoduto.to_sql('Instalacao_Gasoduto', conn, if_exists='append', index=False)

df_instalacao_gasoduto.head()

,codigo_instalacao_gasoduto,nome_instalacao_gasoduto,tipo_instalacao,municipio,uf,codigo_instalacao_transporte
7,111567.0,Cáceres,Ponto de Recebimento,Cáceres,MT,700521
12,111565.0,Termocuiabá,Ponto de Entrega,Cuiabá,MT,700521
29,111566.0,MTGAS,Ponto de Entrega,Cuiabá,MT,700521
41,222297.0,Interconexão Campos Elíseos I (EDG Campos Elís...,Ponto de Recebimento,Duque de Caxias,RJ,700543
46,222296.0,Interconexão TECAB (TECAB >> GASDUC III),Ponto de Recebimento,Macaé,RJ,700543


In [157]:
verify_database_normalization(DB_FILE,'Instalacao_Transporte')

The table 'Instalacao_Transporte' is in 1NF.
The table 'Instalacao_Transporte' is in 2NF.
The table 'Instalacao_Transporte' is in 3NF.


{'Instalacao_Transporte': {'1NF': True, '2NF': True, '3NF': True}}

## Contrato

In [158]:
df_instalacao_transporte = df_long[['codigo_instalacao_transporte', 'nome_instalacao_transporte']] \
    .dropna(subset=['codigo_instalacao_transporte', 'nome_instalacao_transporte']) \
    .drop_duplicates(subset=['nome_instalacao_transporte'])

In [ ]:
df_contrato = df_long[['codigo_instalacao_transporte', 'codigo_carregador', 'nome_contrato']] \
    .dropna(subset=['codigo_instalacao_transporte', 'codigo_carregador','nome_contrato']) \
    .drop_duplicates(subset=['codigo_instalacao_transporte', 'codigo_carregador','nome_contrato'])
# Verifica duplicados com base na chave única da tabela (transporte + carregador)
df_contrato.to_sql('Contrato', conn, if_exists='append', index=False)
df_contrato.head()

,codigo_instalacao_transporte,codigo_carregador,nome_contrato
7,700521,1645009.0,Gasocidente do Mato Grosso Ltda x AMBAR Energi...
24,700521,6023921.0,Gasocidente do Mato Grosso Ltda x Companhia Ma...
41,700543,33000167.0,GASDUC III
93,700525,33000167.0,Paulínia-Jacutinga
110,700540,33000167.0,GASTAU


In [102]:
np.unique(df['tipo_instalacao'].to_list(), return_counts=True)

(array(['Ponto de Entrega', 'Ponto de Recebimento', 'nan'], dtype='<U32'),
 array([6917, 1898, 2324]))

In [103]:
df.dropna(subset=['codigo_carregador'])[['codigo_instalacao_transporte','codigo_carregador' ,'nome_contrato']].drop_duplicates(subset=['codigo_instalacao_transporte','codigo_carregador'])

,codigo_instalacao_transporte,codigo_carregador,nome_contrato
0,700521,1.645009e+06,NaN
17,700521,6.023921e+06,NaN
34,700543,3.300017e+07,NaN
86,700525,3.300017e+07,NaN
103,700540,3.300017e+07,NaN
...,...,...,...
10639,700503,8.686454e+07,SULGAS
10660,700503,3.004424e+09,ENEVA
10772,700503,7.230012e+07,SCGAS
11105,505444,2.072300e+09,NaN


In [ ]:
verify_database_normalization(DB_FILE,'Instalacao_Transporte')

The table 'Contrato' is in 1NF.
The table 'Contrato' has foreign key constraints, which may indicate partial dependencies.


{'Contrato': {'1NF': True, '2NF': False, '3NF': False}}

## VERIfica BD

In [ ]:
import sqlite3
import pandas as pd

# Step 1: Connect to your database
conn = sqlite3.connect(DB_FILE)  # Replace with your actual file

# Step 2: Get the list of all table names
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

# Step 3: Print rows and columns for each table
for table in tables:
    table_name = table[0]
    try:
        # Read table into pandas DataFrame
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        
        # Count rows and columns
        num_rows = df.shape[0]
        num_cols = df.shape[1]

        # Print info
        print(f"Table: {table_name}")
        print(f"  Rows: {num_rows}")
        print(f"  Columns: {num_cols}")
        print("-" * 30)
        
    except Exception as e:
        print(f"Could not read table '{table_name}': {e}")

# Step 4: Close connection
conn.close()


Table: Operador
  Rows: 5
  Columns: 2
------------------------------
Table: Carregador
  Rows: 32
  Columns: 2
------------------------------
Table: Variavel
  Rows: 27
  Columns: 3
------------------------------
Table: sqlite_sequence
  Rows: 3
  Columns: 2
------------------------------
Table: Instalacao_Transporte
  Rows: 34
  Columns: 3
------------------------------
Table: Instalacao_Gasoduto
  Rows: 199
  Columns: 6
------------------------------
Table: Contrato
  Rows: 198
  Columns: 4
------------------------------
Table: Medicao
  Rows: 170355
  Columns: 7
------------------------------


In [ ]:
import sqlite3
import pandas as pd

# Nome do seu arquivo de banco de dados
db_file = 'gas_data.db'

# Conectar ao banco de dados
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Query para listar todas as tabelas no banco de dados
query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
cursor.execute(query_tables)

# Obter os nomes das tabelas
table_names = [table[0] for table in cursor.fetchall()]

print("Tabelas encontradas no banco de dados:")
print(table_names)

# Fechar a conexão
# conn.close() # Manteremos aberta para os próximos passos

Tabelas encontradas no banco de dados:
['Operador', 'Carregador', 'Variavel', 'sqlite_sequence', 'Instalacao_Transporte', 'Instalacao_Gasoduto', 'Contrato', 'Medicao']


In [ ]:
conn = sqlite3.connect('gas_data.db')

# Step 4: Close connection
df = pd.read_sql_query(f"SELECT * FROM Operador", conn)
print(f"Table: Operador")
df

Table: Operador


,codigo_operador,nome_operador
0,5,Transportadora Brasileira Gasoduto Bolívia-Bra...
1,1717813,Gasocidente do Mato Grosso Ltda - GOM
2,3003146349,Transportadora Sulbrasileira de Gás S.A. - TSB
3,3004992714,Nova Transportadora do Sudeste S.A. - NTS
4,3006248349,Transportadora Associada de Gás S.A. - TAG


In [ ]:
# Step 4: Close connection
df = pd.read_sql_query(f"SELECT * FROM Carregador", conn)
print(f"Table: Carregador")
df

Table: Carregador


,codigo_carregador,nome_carregador
0,371600,COMPANHIA PARAIBANA DE GAS PBGAS
1,535681,COMPAGAS
2,1645009,AMBAR
3,2232025,BRAVA
4,2857854,3R PETROLEUM OFFSHORE S.A
5,4423567,ENEVA S.A.
6,6023921,MTGAS
7,16974249,GALP
8,33000167,Petróleo Brasileiro S.A. - PETROBRAS
9,34186669,ORIGEM


In [ ]:
# Step 4: Close connection
df = pd.read_sql_query(f"SELECT * FROM Variavel", conn)
print(f"Table: Variavel")
df

Table: Variavel


,id_variavel,nome_variavel,unidade_medida
0,1,Gás de Uso no Sistema,mil m³
1,2,Gás não contado,mil m³
2,3,Perdas Operacionais,mil m³
3,4,Perdas Extraordinárias,mil m³
4,5,Desequilíbrio Diário,mil m³
5,6,Desequilíbrio Diário Acumulado,mil m³
6,7,Empacotamento,mil m³
7,8,Volume Solicitado,mil m³
8,9,Volume Programado,mil m³
9,10,Volume Realizado,mil m³


In [ ]:
# Step 4: Close connection
df = pd.read_sql_query(f"SELECT * FROM sqlite_sequence", conn)
print(f"Table: sqlite_sequence")
df

Table: sqlite_sequence


,name,seq
0,Variavel,27
1,Contrato,198
2,Medicao,170355


In [ ]:
# Step 4: Close connection
df = pd.read_sql_query(f"SELECT * FROM Instalacao_Gasoduto", conn)
print(f"Table: Instalacao_Gasoduto")
df

Table: Instalacao_Gasoduto


,codigo_instalacao_gasoduto,nome_instalacao_gasoduto,tipo_instalacao,municipio,uf,codigo_instalacao_transporte
0,18368,Corumbá (Mutun),Ponto de Recebimento,Corumbá,MS,700503
1,19581,Corumbá,Ponto de Entrega,Corumbá,MS,700503
2,26983,Campo Grande,Ponto de Entrega,Campo Grande,MS,700503
3,26987,Três Lagoas / UTE,Ponto de Entrega,Três Lagoas,MS,700503
4,26989,Valparaíso,Ponto de Entrega,Valparaíso,SP,700503
...,...,...,...,...,...,...
194,614302,UTE SERGIPE,Ponto de Entrega,Barra dos Coqueiros,SE,702089
195,622774,INTERCONEXÃO CABIÚNAS (GASCAV >> GASDUC),Ponto de Entrega,Macaé,RJ,700539
196,622776,INTERCONEXÃO CABIÚNAS (GASDUC >> GASCAV),Ponto de Recebimento,Macaé,RJ,700539
197,622844,INTERCONEXÃO CABIÚNAS (GASDUC >> GASCAV),Ponto de Entrega,Macaé,RJ,700543


In [ ]:
# Step 4: Close connection
df = pd.read_sql_query(f"SELECT * FROM Contrato", conn)
print(f"Table: Contrato")
df

Table: Contrato


,id_contrato,nome_contrato,codigo_instalacao_transporte,codigo_carregador
0,1,Gasocidente do Mato Grosso Ltda x AMBAR Energi...,700521,1645009
1,2,Gasocidente do Mato Grosso Ltda x Companhia Ma...,700521,6023921
2,3,GASDUC III,700543,33000167
3,4,Paulínia-Jacutinga,700525,33000167
4,5,GASTAU,700540,33000167
...,...,...,...,...
193,194,ENEVA,700503,3004423567
194,195,COMPAGAS,700503,535681
195,196,SCGAS,700503,72300122
196,197,Fronteira (Argentina) - Uruguaiana (Trecho 1),505444,2072300122


In [ ]:
# Step 4: Close connection
df = pd.read_sql_query(f"SELECT * FROM Medicao", conn)
print(f"Table: Medicao")
df

Table: Medicao


,id_medicao,data_medicao,valor,id_variavel,codigo_carregador,codigo_instalacao_transporte,codigo_instalacao_gasoduto
0,1,2025-03-01 00:00:00,0.00,1,1645009,700521,NaN
1,2,2025-03-01 00:00:00,0.00,2,1645009,700521,NaN
2,3,2025-03-01 00:00:00,0.00,3,1645009,700521,NaN
3,4,2025-03-01 00:00:00,0.00,4,1645009,700521,NaN
4,5,2025-03-01 00:00:00,0.00,5,1645009,700521,NaN
...,...,...,...,...,...,...,...
170350,170351,2025-03-31 00:00:00,800.00,8,2072300122,505445,212588.0
170351,170352,2025-03-31 00:00:00,800.00,9,2072300122,505445,212588.0
170352,170353,2025-03-31 00:00:00,459.28,10,2072300122,505445,212588.0
170353,170354,2025-03-31 00:00:00,100.00,11,2072300122,505445,212588.0


In [ ]:
df.isna().sum()

id_medicao                          0
data_medicao                        0
valor                               0
id_variavel                         0
codigo_carregador                   0
codigo_instalacao_transporte        0
codigo_instalacao_gasoduto      11204
dtype: int64